# Model Training: Mistral-7B & Qwen LoRA Fine-Tuning

This notebook trains both MAS models on the synthetic Kubernetes incident data:

- **Model 1 (Mistral-7B)**: RCA classification — structured signals → root cause label
- **Model 2 (Qwen-7B)**: RCA generation — evidence text → diagnosis + fix plan

Both use **LoRA** (Low-Rank Adaptation) for parameter-efficient fine-tuning.

In [ ]:
import torch
import pandas as pd
from pathlib import Path

# Check device
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    print("Device: Apple MPS (Metal Performance Shaders)")
else:
    print("Device: CPU (training will be slow)")

## 1. Verify Training Data

In [ ]:
# Agent 1 data — structured features for Mistral
agent1_df = pd.read_parquet("data/processed/agent1_structured.parquet")
print(f"Agent 1 (structured): {len(agent1_df):,} rows, {len(agent1_df.columns)} cols")
print(f"\nTarget distribution (root_cause_family):")
print(agent1_df["root_cause_family"].value_counts())
agent1_df.head(3)

In [ ]:
# Agent 2 data — evidence text for Qwen
agent2_df = pd.read_parquet("data/processed/agent2_evidence.parquet")
print(f"Agent 2 (evidence): {len(agent2_df):,} rows, {len(agent2_df.columns)} cols")
print(f"\nScenario distribution:")
print(agent2_df["scenario_id"].value_counts().head(10))
print(f"\nSample evidence_text (first 500 chars):")
print(agent2_df["evidence_text"].iloc[0][:500])

---
## 2. Model 1: Mistral-7B LoRA — RCA Classification

Fine-tunes Mistral-7B-Instruct to classify root cause family from structured K8s signals.

| Parameter | Value |
|-----------|-------|
| Base model | `mistralai/Mistral-7B-Instruct-v0.3` |
| LoRA rank | 16 |
| LoRA alpha | 32 |
| Target modules | q_proj, k_proj, v_proj, o_proj |
| Max sequence length | 512 |
| Quantization | 4-bit NF4 (CUDA) / float16 (MPS) |

In [ ]:
from agents.models.model1_mistral_lora import run as run_model1

run_model1(
    input_path=Path("data/processed/agent1_structured.parquet"),
    outdir=Path("agents/models/trained/model1_mistral"),
    epochs=3,
    batch_size=4,
)

### Model 1 — Test Inference

Load the trained adapter and classify a sample incident.

In [ ]:
from agents.models.model1_mistral_lora import predict, load_model_and_tokenizer, MODEL_ID
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load base model + trained LoRA adapter
adapter_path = "agents/models/trained/model1_mistral/lora_adapter"
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

if torch.cuda.is_available():
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                     bnb_4bit_compute_dtype=torch.float16)
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config,
                                                      device_map="auto", torch_dtype=torch.float16)
else:
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="mps" if torch.backends.mps.is_available() else "cpu",
                                                      torch_dtype=torch.float16)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()
print("Model loaded with LoRA adapter.")

In [ ]:
# Test on a sample from the dataset
sample = agent1_df.iloc[42].to_dict()
print(f"Ground truth: {sample['root_cause_family']}")
print(f"Scenario:     {sample['scenario_id']}")
print(f"Pod status:   {sample['pod_status']}")
print(f"Event reason: {sample['event_reason']}")
print()

prediction = predict(model, tokenizer, sample)
print(f"Predicted:    {prediction}")

---
## 3. Model 2: Qwen-7B LoRA — RCA Generation

Fine-tunes Qwen2.5-7B-Instruct to generate diagnosis and remediation from raw evidence.

| Parameter | Value |
|-----------|-------|
| Base model | `Qwen/Qwen2.5-7B-Instruct` |
| LoRA rank | 16 |
| LoRA alpha | 32 |
| Target modules | q/k/v/o_proj + gate/up/down_proj |
| Max sequence length | 1024 |
| Quantization | 4-bit NF4 (CUDA) / float16 (MPS) |

In [ ]:
from agents.models.model2_qwen_lora import run as run_model2

run_model2(
    input_path=Path("data/processed/agent2_evidence.parquet"),
    outdir=Path("agents/models/trained/model2_qwen"),
    epochs=3,
    batch_size=2,
)

### Model 2 — Test Inference

Load the trained adapter and generate a diagnosis for a sample incident.

In [ ]:
from agents.models.model2_qwen_lora import predict as predict_qwen, MODEL_ID as QWEN_MODEL_ID
from peft import PeftModel

# Load base model + trained LoRA adapter
adapter_path_qwen = "agents/models/trained/model2_qwen/lora_adapter"
tokenizer_qwen = AutoTokenizer.from_pretrained(adapter_path_qwen)

if torch.cuda.is_available():
    base_model_qwen = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_ID, quantization_config=bnb_config,
        device_map="auto", torch_dtype=torch.float16, trust_remote_code=True)
else:
    base_model_qwen = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_ID, device_map="mps" if torch.backends.mps.is_available() else "cpu",
        torch_dtype=torch.float16, trust_remote_code=True)

model_qwen = PeftModel.from_pretrained(base_model_qwen, adapter_path_qwen)
model_qwen.eval()
print("Qwen model loaded with LoRA adapter.")

In [ ]:
# Test on a sample from the dataset
sample2 = agent2_df.iloc[42]
print(f"Scenario: {sample2['scenario_id']}")
print(f"Difficulty: {sample2['difficulty']}")
print(f"\n--- Ground Truth Diagnosis ---")
print(sample2["diagnosis_text"][:300])
print(f"\n--- Model Prediction ---")

generated = predict_qwen(
    model_qwen, tokenizer_qwen,
    evidence_text=sample2["evidence_text"],
    scenario_id=sample2["scenario_id"],
    difficulty=sample2["difficulty"],
)
print(generated)

---
## 4. Results Summary

Compare training metrics from both models.

In [ ]:
import json

print("=" * 60)
print("MODEL TRAINING SUMMARY")
print("=" * 60)

for name, path in [("Model 1 (Mistral)", "agents/models/trained/model1_mistral"),
                    ("Model 2 (Qwen)", "agents/models/trained/model2_qwen")]:
    print(f"\n--- {name} ---")
    
    meta_path = Path(path) / ("model1_metadata.json" if "model1" in path else "model2_metadata.json")
    if meta_path.exists():
        with open(meta_path) as f:
            meta = json.load(f)
        print(f"  Base model:    {meta.get('model_id')}")
        print(f"  Task:          {meta.get('task')}")
        print(f"  Dataset rows:  {meta.get('dataset_rows'):,}")
        print(f"  Epochs:        {meta.get('epochs')}")
        print(f"  LoRA rank:     {meta.get('lora_r')}")
    
    eval_path = Path(path) / "eval_metrics.json"
    if eval_path.exists():
        with open(eval_path) as f:
            metrics = json.load(f)
        print(f"  Eval loss:     {metrics.get('eval_loss', 'N/A'):.4f}")
    else:
        print("  (eval metrics not yet available — run training first)")